# Benchmarks for Microgrid environment

The notebook provides testing and comparison between deterministic policies and a pretrained RL agent strategies among the `MicroGridEnv` environment.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%cd ../..

/home/andreabarisione/ErNESTO-gym


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime
from tqdm import tqdm
from collections import OrderedDict, defaultdict
from gymnasium.utils.env_checker import check_env
import gymnasium as gym

from ernestogym.envs.single_agent.env_new import MicroGridEnv
from ernestogym.envs.single_agent.utils import parameter_generator

In [3]:
sns.set_style('darkgrid')
plot_colors = sns.color_palette()
sns.set(font_scale=1.2)

alg_color = OrderedDict({
    'random': plot_colors[0],
    'only_market': plot_colors[1],
    'battery_first': plot_colors[2],
    '50-50': plot_colors[3],
    'ppo': plot_colors[4]
})

alg_markers = OrderedDict({
    'random': '.',
    'only_market': 'o',
    'battery_first': 'v',
    '50-50': 'P',
    'ppo': '*',
})

alg_labels = {
    'random': 'Random',
    'only_market': 'OM',
    'battery_first': 'BF',
    '50-50': '50-50',
    'ppo': 'PPO',
}

In [ ]:
# IF RESULTS ARE ALREADY AVAILABLE, GO TO LAST CELLS TO LOAD THEM FROM JSON FILE AND DIRECTLY PLOT THEM!!

## Deterministic policies
Here we can evaluate different rule-based and deterministic policies.
Hereafter we will test:
1. random action policy
2. market-only policy
3. battery-first policy
5. 50/50 policy

In [4]:
params = parameter_generator(world_options='ernestogym/envs/single_agent/ijcnn_deg_test_cell.yaml')
env = gym.make(id="ernestogym/micro_grid-v1", **{'settings':params})

++++++ INITIALIZING ENV +++++++
[INIT] Environment created with seed 1234


In [5]:
params['battery']['params']['nominal_cost'] = 10.5
params

{'battery': {'sign_convention': 'passive',
  'params': {'nominal_voltage': 3.65,
   'nominal_capacity': 20.0,
   'nominal_dod': 0.8,
   'nominal_lifetime': 3000,
   'v_max': 4.15,
   'v_min': 3.0,
   'temp_ambient': 298.15,
   'nominal_cost': 10.5},
  'bounds': {'voltage': {'low': 3.0, 'high': 4.15},
   'current': {'low': -60.0, 'high': 20.0},
   'power': {'low': -249.0, 'high': 83},
   'temperature': {'low': 273.15, 'high': 323.15},
   'temp_ambient': {'low': 273.15, 'high': 313.15},
   'soc': {'low': 0.1, 'high': 1.0},
   'soh': {'low': 0.6, 'high': 1.0}},
  'init': {'voltage': 3.0,
   'current': 0.0,
   'power': 0.0,
   'temperature': 293.15,
   'temp_ambient': 293.15,
   'soc': 1.0,
   'soh': 1.0}},
 'input_var': 'power',
 'models_config': [{'type': 'electrical',
   'class_name': 'TheveninModel',
   'use_fading': False,
   'components': {'r0': {'selected_type': 'lookup',
     'scalar': 10.0,
     'lookup': {'table': 'r0_cell.csv',
      'inputs': [{'var': 'temperature', 'label': 't

In [6]:
# Test profiles belonging to the test set
test_profiles = [i for i in range(370, 398)]
rewards = defaultdict(dict)

<h3> Random Policy: </h3>
The action is chosen randomly at each decision step.

In [ ]:
alg = 'random'
rewards[alg] = {}

for profile in tqdm(test_profiles):
    obs, info = env.reset(options={'eval_profile': str(profile)})
    done = False
    rewards[alg][profile] = {}
    rewards[alg][profile]['pure'] = []

    while not done:
        action = env.action_space.sample()  # Randomly select an action
        obs, reward, terminated, truncated, info = env.step(action)  
        done = terminated or truncated
        rewards[alg][profile]['pure'].append(list(info['pure_rewards'].values()))

<h3> Only Market Policy: </h3>
The action chosen is always 0, meaning that the battery is never used.

In [ ]:
alg = 'only_market'
rewards[alg] = {}

for profile in tqdm(test_profiles):
    obs, info = env.reset(options={'eval_profile': str(profile)})
    done = False
    rewards[alg][profile] = {}
    rewards[alg][profile]['pure'] = []

    while not done:
        action = np.array([0]) # Only trading with market
        obs, reward, terminated, truncated, info = env.step(action)  
        done = terminated or truncated
        rewards[alg][profile]['pure'].append(list(info['pure_rewards'].values()))

<h3> Battery First Policy: </h3>
The action chosen is always 1, meaning that the battery is always used before interacting with the market.

In [ ]:
alg = 'battery_first'
rewards[alg] = {}

for profile in tqdm(test_profiles):
    obs, info = env.reset(options={'eval_profile': str(profile)})
    done = False
    rewards[alg][profile] = {}
    rewards[alg][profile]['pure'] = []

    while not done:
        action = np.array([1])  # Use the battery as much as possible 
        obs, reward, terminated, truncated, info = env.step(action)  
        done = terminated or truncated
        rewards[alg][profile]['pure'].append(list(info['pure_rewards'].values()))

<h3> 50-50 Policy: </h3>
The action chosen is always 0.5, meaning that the battery is never used.

In [ ]:
alg = '50-50'
rewards[alg] = {}

for profile in tqdm(test_profiles):
    obs, info = env.reset(options={'eval_profile': str(profile)})
    done = False
    rewards[alg][profile] = {}
    rewards[alg][profile]['pure'] = []

    while not done:
        action = np.array([0.5])  # Use the battery at 50%
        obs, reward, terminated, truncated, info = env.step(action)  
        done = terminated or truncated
        rewards[alg][profile]['pure'].append(list(info['pure_rewards'].values()))

### PPO agent
Here we load the previously created model `PPO_trained`.

In [ ]:
# Uncomment the following line to install stable-baselines3
#!pip install stable-baselines3

In [7]:
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy
from stable_baselines3.common.env_util import make_vec_env

In [8]:
env = make_vec_env("ernestogym/micro_grid-v1", n_envs=1, env_kwargs={'settings':params})

alg = 'ppo'
rewards[alg] = {}

model = PPO(MlpPolicy, env, verbose=1)
vec_env = model.get_env()
# model = PPO.load("examples/single_agent/models/Weekly_experiment_longer_20250811_seed11.zip")
model = PPO.load("examples/single_agent/models/model_20251111.zip")


for profile in tqdm(test_profiles):
    vec_env.set_options({'eval_profile': str(profile)})
    obs = vec_env.reset()

    cumulated_reward = 0
    rewards[alg][profile] = {}
    rewards[alg][profile]['pure'] = []
    done = False
    
    while not done:
        action, _states = model.predict(obs)
        obs, r, dones, info = vec_env.step(action)
        done = dones[0]
        rewards[alg][profile]['pure'].append(list(info[0]['pure_rewards'].values()))

/home/andreabarisione/miniconda3/envs/ernesto-gym/lib/python3.11/site-packages/gymnasium/envs/registration.py:787: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes ([]).
  logger.warn(


++++++ INITIALIZING ENV +++++++
[INIT] Environment created with seed 1234
Using cpu device


  0%|          | 0/28 [00:00<?, ?it/s]

[0.] {'temperature': 278.34, 'soc': 1.0, 'demand': 5.614583333, 'generation': 0.0, 'ask': 0.0001341, 'bid': 4.71e-05, 'sin_day_of_year': 0.8111197574030781, 'cos_day_of_year': 0.5848801066461158, 'sin_seconds_of_day': -0.25881904510252146, 'cos_seconds_of_day': 0.9659258262890681}
[0.] {'temperature': 278.99894775358115, 'soc': 1, 'demand': 4.586805556, 'generation': 0.0, 'ask': 0.0001287286, 'bid': 4.17286e-05, 'sin_day_of_year': 0.811539059007361, 'cos_day_of_year': 0.5842981736283684, 'sin_seconds_of_day': 7.84516728218212e-15, 'cos_seconds_of_day': 1.0}
[0.01390678] {'temperature': 279.15525889336203, 'soc': 1, 'demand': 2.489583333, 'generation': 0.0, 'ask': 0.00012425, 'bid': 3.725e-05, 'sin_day_of_year': 0.811957943107363, 'cos_day_of_year': 0.5837159400126573, 'sin_seconds_of_day': 0.2588190451024817, 'cos_seconds_of_day': 0.9659258262890787}
[0.] {'temperature': 279.19233294264893, 'soc': 1, 'demand': 0.763888889, 'generation': 0.0, 'ask': 0.000124, 'bid': 3.7e-05, 'sin_day_of

  4%|▎         | 1/28 [00:01<00:30,  1.15s/it]

...

[0.] {'temperature': 285.7909771835338, 'soc': 1, 'demand': 0.777777778, 'generation': 0.208333333, 'ask': 0.00012928123, 'bid': 4.228123e-05, 'sin_day_of_year': -0.8561806254157887, 'cos_day_of_year': 0.5166766267818093, 'sin_seconds_of_day': 0.9659258262890923, 'cos_seconds_of_day': -0.2588190451024311}
[0.] {'temperature': 285.7909758395103, 'soc': 1, 'demand': 2.611111111, 'generation': 0.90625, 'ask': 0.00013103431, 'bid': 4.403431e-05, 'sin_day_of_year': -0.8558098144599982, 'cos_day_of_year': 0.5172905967383744, 'sin_seconds_of_day': 0.8660254037845092, 'cos_seconds_of_day': -0.4999999999998778}
[0.] {'temperature': 285.79097552075046, 'soc': 1, 'demand': 1.760416667, 'generation': 1.71875, 'ask': 0.000132, 'bid': 4.5e-05, 'sin_day_of_year': -0.8554385632243997, 'cos_day_of_year': 0.5179043005696851, 'sin_seconds_of_day': 0.7071067811866815, 'cos_seconds_of_day': -0.7071067811864136}
[0.05661849] {'temperature': 285.79097544515065, 'soc': 1, 'demand': 2.871527778, 'generation': 

 96%|█████████▋| 27/28 [00:34<00:01,  1.49s/it]

[0.] {'temperature': 278.406766374189, 'soc': 1, 'demand': 0.361111111, 'generation': 0.0, 'ask': 0.000129, 'bid': 4.2e-05, 'sin_day_of_year': -0.012910296075008904, 'cos_day_of_year': 0.9999166586547379, 'sin_seconds_of_day': 1.0, 'cos_seconds_of_day': -4.7563957315934604e-14}
[0.] {'temperature': 278.4068316713314, 'soc': 1, 'demand': 0.020833333, 'generation': 0.083333333, 'ask': 0.00013340779, 'bid': 4.640779e-05, 'sin_day_of_year': -0.012193093996176385, 'cos_day_of_year': 0.9999256614662914, 'sin_seconds_of_day': 0.9659258262891273, 'cos_seconds_of_day': -0.25881904510230047}
[0.] {'temperature': 278.40684715775024, 'soc': 1, 'demand': 0.211805556, 'generation': 1.427083333, 'ask': 0.00013288453, 'bid': 4.588453e-05, 'sin_day_of_year': -0.011475885644485396, 'cos_day_of_year': 0.9999341498562166, 'sin_seconds_of_day': 0.8660254037844631, 'cos_seconds_of_day': -0.4999999999999576}
[0.05754024] {'temperature': 278.4068508306394, 'soc': 1, 'demand': 0.440972222, 'generation': 3.1979

100%|██████████| 28/28 [00:36<00:00,  1.30s/it]

[0.] {'temperature': 289.9900961032852, 'soc': 1, 'demand': 0.28125, 'generation': 0.0, 'ask': 0.00012476833, 'bid': 3.776833e-05, 'sin_day_of_year': -0.907616955790474, 'cos_day_of_year': 0.41979931105426177, 'sin_seconds_of_day': -0.5000000000001829, 'cos_seconds_of_day': 0.866025403784333}
[0.06831742] {'temperature': 289.9900961032852, 'soc': 1, 'demand': 0.211805556, 'generation': 0.0, 'ask': 0.0001259743, 'bid': 3.89743e-05, 'sin_day_of_year': -0.9073156176852937, 'cos_day_of_year': 0.420450199077553, 'sin_seconds_of_day': -0.2588190451025517, 'cos_seconds_of_day': 0.96592582628906}
[0.] {'temperature': 290.0465473100488, 'soc': 1, 'demand': 0.336805556, 'generation': 0.0, 'ask': 0.00012364, 'bid': 3.664e-05, 'sin_day_of_year': -0.9070138128026362, 'cos_day_of_year': 0.42110087079608916, 'sin_seconds_of_day': -8.033930594661272e-14, 'cos_seconds_of_day': 1.0}
[0.] {'temperature': 290.05993571216476, 'soc': 1, 'demand': 0.222222222, 'generation': 0.0, 'ask': 0.00012236585, 'bid': 

In [ ]:
env

In [ ]:
#import json

#with open('examples/microgrid/test_results.json', 'w') as f:
#    json.dump(rewards, f, indent=4)

## Result analysis
Here we compare the average cumulated reward across all the test profiles among the different methods.

In [ ]:
#import json

#with open('examples/microgrid/test_results.json', 'r') as f:
#    rewards = json.load(f)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4), tight_layout=True)

for i, alg in enumerate(rewards.keys()):
    cum_rewards = []
    for profile in rewards[alg].keys():
        cum_rewards.append(np.cumsum([rewards[alg][profile]['pure'][i][0] + rewards[alg][profile]['pure'][i][1] + rewards[alg][profile]['pure'][i][2] for i in range(len(rewards[alg][profile]['pure']))]))
    
    means = np.mean(cum_rewards, axis=0)
    stds = np.std(cum_rewards, axis=0)
    ci = 1.96 * stds/np.sqrt(len(rewards[alg].keys()))
    
    ax.plot(means, label=alg_labels[alg], color=alg_color[alg])        
    ax.fill_between(range(len(means)), means + ci, means - ci, color=alg_color[alg], alpha=0.1)
    ax.legend()

In [ ]:
box_data = {}
colors = []

for alg in rewards.keys():
    box_data[alg_labels[alg]] = []
    
    for profile in rewards[alg].keys():
        total_trad = np.sum([rewards[alg][profile]['pure'][i][0] for i in range(len(rewards[alg][profile]['pure']))])
        total_deg = np.sum([rewards[alg][profile]['pure'][i][1] for i in range(len(rewards[alg][profile]['pure']))])
        box_data[alg_labels[alg]].append(total_trad + total_deg)

    colors.append(alg_color[alg])
        
fig, ax = plt.subplots(figsize=(10, 4), tight_layout=True)
box_plot = sns.boxplot(box_data, gap=.1, palette=colors, width=.8)

medians = [np.mean(values) for key, values in box_data.items()]
vertical_offset = -130 # offset from median for display

for xtick, alg in zip(box_plot.get_xticks(), rewards.keys()):
    box_plot.text(xtick, 
                  vertical_offset, 
                  round(medians[xtick]), 
                  horizontalalignment='center',
                  size='x-small', 
                  color=alg_color[alg], 
                  weight='semibold')